In [ ]:
from collections import defaultdict

# Эмуляция "Распределенной файловой системы" (HDFS)
# Представим, что эти 3 строки лежат на 3-х разных физических серверах
HDFS_DATA = [
    "error timeout database error",     # Сервер 1
    "database connection success",      # Сервер 2
    "error timeout success success"     # Сервер 3
]

# ========================================================
# ФАЗА 1: MAP (Отображение)
# Выполняется ЛОКАЛЬНО на каждом из 3-х серверов. 
# Мы не гоняем терабайты текста по сети! Мы отправляем КОД к данным.
# ========================================================
def mapper(text_chunk: str) -> list:
    """Берет кусок текста и выдает пары (Ключ, 1)"""
    mapped_data = []
    # Эмуляция токенизации (в реальности тут может быть сложный парсинг логов)
    words = text_chunk.split()
    for word in words:
        mapped_data.append((word, 1))
    return mapped_data

# ========================================================
# ФАЗА 2: SHUFFLE (Перемешивание) - САМАЯ ТЯЖЕЛАЯ ФАЗА!
# Здесь данные летят по сети. Все одинаковые ключи со всех серверов
# должны собраться на одном конкретном сервере-Reduce.
# ========================================================
def shuffle(mapped_results: list) -> dict:
    """Группирует все единички по ключам"""
    shuffled_data = defaultdict(list)
    
    # mapped_results - это список списков от всех мапперов
    for node_result in mapped_results:
        for key, value in node_result:
            shuffled_data[key].append(value)
            
    return shuffled_data

# ========================================================
# ФАЗА 3: REDUCE (Свертка)
# Выполняется на агрегирующих серверах.
# Берет ключ и огромный массив единичек -> возвращает одно число (сумму).
# ========================================================
def reducer(key: str, values_list: list) -> tuple:
    """Схлопывает массив значений [1, 1, 1...] в сумму"""
    return (key, sum(values_list))


if __name__ == "__main__":
    print("--- ЗАПУСК РАСПРЕДЕЛЕННОГО MAP-REDUCE ---\n")
    
    print("[1. MAP PHASE] Серверы парсят свои куски текста...")
    map_outputs = []
    for i, data_chunk in enumerate(HDFS_DATA, 1):
        result = mapper(data_chunk)
        map_outputs.append(result)
        print(f"   Узел {i} выдал: {result}")
        
    print("\n[2. SHUFFLE PHASE] Группировка по сети (Network I/O)...")
    shuffled = shuffle(map_outputs)
    for k, v in shuffled.items():
        print(f"   Ключ '{k}' собрал массив: {v}")
        
    print("\n[3. REDUCE PHASE] Финальная агрегация...")
    final_output = []
    for key, values in shuffled.items():
        # В реальности ключи раскидываются по разным Reduce-серверам
        final_result = reducer(key, values)
        final_output.append(final_result)
        
    print("\n--- ИТОГОВЫЙ РЕЗУЛЬТАТ ---")
    for res in sorted(final_output, key=lambda x: x[1], reverse=True):
        print(f"{res[0]}: {res[1]}")
